# General inference and priors

This tutorial uses the general inference framework: no profile-specific fitting function is required. A **prior** describes our probability distribution for a parameter before seeing the image. Combining all priors with the imaging likelihood produces the posterior. We will use uniform, Gaussian, log-uniform, and truncated-Gaussian priors in one model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from akira import (
    CartesianGrid, CompositeLight, EllipticalPowerLaw, ExternalShear,
    FreeParameter, GaussianPrior, LensModel, LensPlane, LogUniformPrior,
    ProfileModel, Sersic, TruncatedGaussianPrior, UniformPrior,
    fit_model, gaussian_psf, render, simulate_imaging,
)

## Simulate a known lens

The simulation uses ordinary scientific profile objects. The parameter declarations are only needed when we decide what to infer.

In [ ]:
grid = CartesianGrid(shape=(80, 80), pixel_scale=0.05, supersampling=2)
true_mass = EllipticalPowerLaw(0.93, slope=2.07, q=0.76, phi=0.28)
true_shear = ExternalShear(0.035, phi=0.61)
true_source = Sersic(
    intensity=0.12, effective_radius=0.16, sersic_index=1.2,
    centre=(0.07, -0.045), q=0.68, phi=-0.22,
)
noiseless = render(
    grid, LensPlane((true_mass, true_shear)), CompositeLight((true_source,))
)
observation = simulate_imaging(
    noiseless, gaussian_psf(0.85, size=7), noise_sigma=0.004,
    pixel_scale=grid.pixel_scale, seed=2026,
)

## Declare fixed and free values

Plain numerical values remain fixed. `FreeParameter(initial, prior)` marks a scalar for inference. Here the same general machinery discovers six free parameters spread across the mass and source profiles. A log-uniform prior is useful for positive scale-like quantities spanning multiplicative ranges.

In [ ]:
model = LensModel(
    lens=(
        ProfileModel(
            EllipticalPowerLaw,
            einstein_radius=FreeParameter(0.84, UniformPrior(0.7, 1.2)),
            slope=FreeParameter(1.95, GaussianPrior(2.0, 0.3)),
            centre=(0.0, 0.0), q=0.76, phi=0.28,
        ),
        ProfileModel(ExternalShear, strength=0.035, phi=0.61),
    ),
    source=(
        ProfileModel(
            Sersic,
            intensity=FreeParameter(0.09, LogUniformPrior(0.02, 0.5)),
            effective_radius=FreeParameter(0.20, LogUniformPrior(0.05, 0.5)),
            sersic_index=1.2,
            centre=(
                FreeParameter(0.02, TruncatedGaussianPrior(0.0, 0.15, -0.3, 0.3)),
                FreeParameter(-0.01, TruncatedGaussianPrior(0.0, 0.15, -0.3, 0.3)),
            ),
            q=0.68, phi=-0.22,
        ),
    ),
)
print('Automatically discovered parameters:')
for name, value in zip(model.parameter_names, model.initial_vector):
    print(f'  {name}: {float(value):.4f}')

## Unit-cube transformation

Every prior converts a number in $[0,1]$ to a physical value. Nested samplers use exactly this operation. The midpoint does not always equal the arithmetic midpoint: for example, a log-uniform prior maps it to the geometric mean.

In [ ]:
median_values = np.asarray(model.from_unit(np.full(len(model.parameter_names), 0.5)))
for name, value in zip(model.parameter_names, median_values):
    print(f'{name}: prior median = {value:.5f}')

## Maximum-a-posteriori fit

`mode='map'` maximizes `log_likelihood + log_prior`. Using `mode='maximum_likelihood'` would retain the prior support as safe parameter bounds but omit prior density from the objective.

In [ ]:
result = fit_model(model, observation, grid, mode='map')
truth = np.array([0.93, 2.07, 0.12, 0.16, 0.07, -0.045])
print(f'optimizer success: {result.success}; evaluations: {result.evaluations}')
print(f'log likelihood: {result.log_likelihood:.2f}')
print(f'log prior:      {result.log_prior:.2f}')
print('\nparameter                                                   true   fitted')
for name, true, fitted in zip(result.parameter_names, truth, result.parameter_vector):
    print(f'{name:58s} {true:7.4f} {float(fitted):8.4f}')

In [ ]:
data = np.asarray(observation.data)
fitted = np.asarray(result.model_image)
residual_sigma = (data - fitted) / np.asarray(observation.noise_map)
half_width = grid.shape[1] * grid.pixel_scale / 2
half_height = grid.shape[0] * grid.pixel_scale / 2
extent = (-half_width, half_width, -half_height, half_height)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, values, title, cmap in zip(
    axes, (data, fitted, residual_sigma),
    ('data', 'general MAP model', 'residual / noise'),
    ('magma', 'magma', 'coolwarm'),
):
    shown = axis.imshow(values, origin='lower', extent=extent, cmap=cmap)
    axis.set_title(title); axis.set_xlabel('x [arcsec]')
    fig.colorbar(shown, ax=axis, fraction=0.046)
axes[0].set_ylabel('y [arcsec]')
fig.tight_layout(); plt.show()

The result also contains the reconstructed best-fitting `LensPlane` and `CompositeLight`, a name-to-value parameter dictionary, an L-BFGS inverse-Hessian covariance approximation, and optimizer diagnostics. This covariance is only a local approximation; posterior sampling will be needed for robust, potentially non-Gaussian uncertainties.